# Calibrated ridge transplant, v9: learned baselines (official AweDist + our TokDist), raw and with calibrated length
Same pipeline and word selection as v6. New: each learned row is also evaluated with the length chosen by the same calibration as ridge and OMP ("(cal)"); AweDist gets 7500 PubMed abstracts (our training abstracts + extra ones never used for evaluation). Tied receivers only; seed 1.
Run the cells in order. AweDist needs internet (git clone, pip).

# Real vocabulary transplant, v5: extra baselines, bits per byte, and larger / untied receivers

Same protocol as v4 (new domain words added as single tokens; zero-shot and after 200 steps training only the new rows). Added in v5, to answer the referee points:

* **New baselines** (all labelled as our own implementations): plain least squares (`LS`, lambda -> 0), orthogonal `Procrustes`, and a SALT-style local regression (`SALT-ours`: for every new token, a ridge regression with intercept fitted on its k nearest shared tokens in the donor space; the original SALT code was not available, so this is an adaptation, not the official method). Official OMP-64 (mergekit `batch_omp`), official WECHSEL (package `wechsel`) and ridge are re-run on the same sentences.
* **Bits per byte**: every result stores the UTF-8 byte count of each sentence, so differences can be reported in bits per byte (comparable across tokenizers and normalised for sentence length).
* **Scale and untied embeddings**: `RUN_SET = "scale"` runs GPT-2 large (tied, 0.77B), Pythia-1B (UNTIED input/output matrices, 1.0B) and Qwen2.5-1.5B (tied). For an untied receiver the whole pipeline is applied twice, once for the input matrix (donor input embeddings -> receiver input embeddings) and once for the output matrix (donor output embeddings, or input embeddings if the donor is tied -> receiver output embeddings). One shared length scale `c` is calibrated for both. Training updates only the new rows of both matrices.

This file runs the **main** set (the four small tied pairs); run all cells. Results go to `transplant_results_v5_main/`.


In [ ]:
%pip install -q transformers datasets scikit-learn
import os
CACHE_ROOT = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
os.environ["XDG_CACHE_HOME"] = CACHE_ROOT + "/cache"      # fastText model goes here (not into the saved output)


In [ ]:
import re, time, gc, random, pickle, traceback, logging, json
from typing import Tuple
from pathlib import Path
from collections import Counter
from types import SimpleNamespace
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
from sklearn.linear_model import orthogonal_mp
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## Official OMP (mergekit batch_omp, verbatim)

In [ ]:
# Official batch_omp, copied verbatim from mergekit 0.1.5 (mergekit/tokensurgeon/omp.py). This is the function mergekit-tokensurgeon calls for '-a omp'.
# Copied (not imported) so that no extra package has to be installed on Kaggle. Only the function body is used.
LOG = logging.getLogger('batch_omp')

def batch_omp(
    targets: torch.Tensor,
    candidate_points: torch.Tensor,
    k: int,
    eps: float = 1e-8,
    reorthogonalize_interval: int = 256,
) -> Tuple[torch.LongTensor, torch.Tensor]:
    """
    Batched Orthogonal Matching Pursuit (OMP) to select `k` points from `candidate_points` that best approximate each target in `targets`.

    Args:
        targets: (B, D) tensor of target vectors.
        candidate_points: (N, D) tensor of candidate points.
        k: Number of points to select (sparsity level).
        eps: Tolerance for numerical stability.
        reorthogonalize_interval: Number of iterations between reorthogonalization steps.

    Returns:
        selected_indices: (B, k) tensor of indices selected for each target.
        coeff: (B, k) tensor of coefficients for each selected point.
    """
    B, D = targets.shape
    N, _ = candidate_points.shape
    device = targets.device
    if k > N:
        raise ValueError(f"Cannot select {k} points from {N} candidates")
    work_dtype = (
        targets.dtype
        if targets.dtype in (torch.float32, torch.float64)
        else torch.float32
    )
    # Convert inputs to work_dtype
    targets_work = targets.to(dtype=work_dtype)
    points_work = candidate_points.to(dtype=work_dtype)
    # Preallocate tensors
    q = torch.zeros((B, D, k), dtype=work_dtype, device=device)
    r = torch.zeros((B, k, k), dtype=work_dtype, device=device)
    selected_indices = torch.zeros((B, k), dtype=torch.long, device=device)
    mask = torch.zeros((B, N), dtype=torch.bool, device=device)
    residuals = targets_work.clone()

    for t in range(k):
        rms_0 = residuals.norm(dim=1).mean()
        # Compute absolute inner products between residuals and points
        abs_inner = (residuals @ points_work.T).abs()  # (B, N)
        # Mask out already selected points
        abs_inner.masked_fill_(mask, -float("inf"))

        # Select new index with maximum correlation
        _, new_idx = torch.max(abs_inner, dim=1)  # (B,)
        selected_indices[:, t] = new_idx
        mask[torch.arange(B, device=device), new_idx] = True

        new_atom = points_work[new_idx]  # (B, D)
        if t == 0:
            r[:, 0, 0] = new_atom.norm(dim=1)
            norm = r[:, 0, 0].clamp(min=eps)
            q[:, :, 0] = new_atom / norm.unsqueeze(1)
        else:
            # Project onto existing basis
            projections = torch.bmm(
                q[:, :, :t].transpose(1, 2), new_atom.unsqueeze(-1)
            ).squeeze(-1)  # (B, t)
            residual = new_atom - torch.bmm(
                q[:, :, :t], projections.unsqueeze(-1)
            ).squeeze(-1)  # (B, D)
            norm = torch.clamp(torch.norm(residual, dim=1), min=eps)
            # Update R and Q
            r[:, :t, t] = projections
            r[:, t, t] = norm
            q[:, :, t] = residual / norm.unsqueeze(-1)

        if t > 0 and t % reorthogonalize_interval == 0:
            q_b = q[:, :, : t + 1]
            q_new, r_new = torch.linalg.qr(q_b, mode="reduced")
            r[:, : t + 1, : t + 1] = torch.bmm(r_new, r[:, : t + 1, : t + 1])
            q[:, :, : t + 1] = q_new

        qt_targets = torch.bmm(
            q[:, :, : t + 1].transpose(1, 2), targets_work.unsqueeze(-1)
        )  # (B, t+1, 1)
        approx = torch.bmm(q[:, :, : t + 1], qt_targets).squeeze(-1)
        residuals = targets_work - approx
        LOG.debug(f"OMP iteration {t}: RMS {rms_0} -> {residuals.norm(dim=1).mean()}")

    # Get final coefficients
    final_coeff = torch.linalg.solve_triangular(
        r[:, :k, :k],
        torch.bmm(q[:, :, :k].transpose(1, 2), targets_work.unsqueeze(-1)),
        upper=True,
    ).squeeze(-1)

    # Print residuals if we're yapping
    if LOG.isEnabledFor(logging.DEBUG):
        rt_approx = torch.bmm(
            final_coeff.unsqueeze(1), points_work[selected_indices]
        ).squeeze(1)
        residuals = targets_work - rt_approx
        LOG.debug(f"OMP final RMS: {residuals.norm(dim=1).mean()}")

    return selected_indices, final_coeff


## Config

In [ ]:
RUN_SET = "learned"         # v6 word selection (training text only) + learned baselines
SEEDS = [1]          # all seeds run one after another in this session (finished runs are skipped on restart)
SEED = SEEDS[0]            # kept for the rest of the notebook
DOMAIN = "pubmed"          # label used in the output file name ("pubmed" or "arxiv"); for arXiv also set DOMAIN_TXT to a file with one abstract per line
OMP64_K = 64                # the official tool's default
N_ABSTRACTS = 4000          # PubMedQA abstracts read (70% train / 30% eval, split by abstract)
TRAIN_FRAC = 0.7
K_TERMS = 300               # maximum number of new tokens added
MIN_TRAIN_COUNT = 5         # a word must occur this often in the TRAINING text (the evaluation text is not used)
MIN_TERMS = 50              # skip a pair with fewer eligible new words
MIN_PIECES = 2              # the receiver must split the word into at least this many tokens
OMP_K = 32
SALT_K = 2000              # neighbours per new token for the SALT-style local regression (our adaptation)
SALT_LAM = 1e-2            # its relative ridge strength
GRID = [0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.5]    # embedding scale, relative to the mean receiver radius
N_TRAIN_T, N_TRAIN_C = 300, 150        # calibration sentences: with new words / without
N_EVAL_T, N_EVAL_C = 600, 200          # evaluation sentences: with new words / each control set
POOL_MAX = 2000                        # training pool for the optional new-row training
TRAIN_STEPS, TRAIN_LR, BATCH = 200, 1e-3, 8
TD_PER_WORD, TD_WINDOW, TD_EPOCHS, TD_BATCH, TD_LRS = 25, 12, 3, 32, [3e-4, 1e-3, 3e-3]   # TokDist (ours): contexts per word, segments each side, epochs, batch, learning rates tried    # 0 = zero-shot only. Training changes ONLY the new rows
EVAL_BATCH = 16
USE_WIKI = True             # WikiText-2 as a general-text control (falls back to domain text only if it cannot load)
DOMAIN_TXT = None           # optional path: one abstract per line, to skip the PubMedQA download
OUT = Path(f"transplant_results_v9_{RUN_SET}"); OUT.mkdir(exist_ok=True)   # new folder: avoids the stale-folder skip of old results
WECHSEL_CACHE = Path(CACHE_ROOT) / "wechsel_matrices"; WECHSEL_CACHE.mkdir(parents=True, exist_ok=True)
MAIN_PAIRS = [   # tied-embedding receivers
    dict(name="GPT-2 <- Pythia-160m",     receiver="openai-community/gpt2", donor="EleutherAI/pythia-160m"),
    dict(name="GPT-2 <- Qwen2.5-0.5B",    receiver="openai-community/gpt2", donor="Qwen/Qwen2.5-0.5B"),
    dict(name="OPT-125m <- Pythia-160m",  receiver="facebook/opt-125m",     donor="EleutherAI/pythia-160m"),
    dict(name="OPT-125m <- Qwen2.5-0.5B", receiver="facebook/opt-125m",     donor="Qwen/Qwen2.5-0.5B"),
]
SCALE_PAIRS = [  # larger receivers; Pythia-1B has separate (untied) input and output matrices
    dict(name="GPT-2 large <- Qwen2.5-0.5B",  receiver="openai-community/gpt2-large", donor="Qwen/Qwen2.5-0.5B"),
    dict(name="Pythia-1B <- Qwen2.5-0.5B",    receiver="EleutherAI/pythia-1b",        donor="Qwen/Qwen2.5-0.5B"),
    dict(name="Qwen2.5-1.5B <- Pythia-410m",  receiver="Qwen/Qwen2.5-1.5B",           donor="EleutherAI/pythia-410m"),
]
PAIRS = MAIN_PAIRS + [p for p in SCALE_PAIRS if "Pythia-1B" not in p["name"]]    # tied receivers only; the four small pairs run first
if RUN_SET == "scale": MIN_TERMS = 30      # larger tokenizers leave fewer words that are split; a pair with fewer than this is skipped

AW_REPO = "https://github.com/konstantinjdobler/awedist"      # official code of Dobler, Elliott & de Melo (arXiv:2505.20133)
AW_LR, AW_SNIPPETS, AW_EPOCHS = 1e-4, 5, 1                   # official default lr 1e-4; snippet_num lowered from 25 to 5 because our training text has few occurrences per word

N_AW_DOCS, AW_STOP = 4000, 3000      # the official script reads 500, 1000, 2000 documents in turn and stops after 3500 (>= AW_STOP); it fails if it runs out of documents, so give it 4000: our training abstracts + extra ones never used for evaluation


In [ ]:
# ---- clone the official AweDist repository and install what its scripts import ----
import subprocess, sys, ast, importlib.util, shutil, re
WORK = Path(CACHE_ROOT) / "awedist_work"; WORK.mkdir(parents=True, exist_ok=True)
REPO = WORK / "awedist"
if not REPO.exists(): subprocess.run(["git", "clone", "--depth", "1", AW_REPO, str(REPO)], check=True)
AW_COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(); print("AweDist commit:", AW_COMMIT)
mods = set()
for p in REPO.glob("*.py"):
    for n in ast.walk(ast.parse(p.read_text())):
        if isinstance(n, ast.Import): mods.update(a.name.split(".")[0] for a in n.names)
        elif isinstance(n, ast.ImportFrom) and n.module and n.level == 0: mods.add(n.module.split(".")[0])
local = {p.stem for p in REPO.glob("*.py")}; PIPNAME = {"sklearn": "scikit-learn", "yaml": "pyyaml", "cv2": "opencv-python"}
missing = sorted(m for m in mods if m not in local and m not in sys.stdlib_module_names and m != "flash_attn" and importlib.util.find_spec(m) is None)
print("modules to install:", missing)
if missing: print(subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + [PIPNAME.get(m, m) for m in missing], capture_output=True, text=True).stderr[-1500:])

# make sure every module the official script imports is present: run it with --help and install whatever is reported missing
for _ in range(12):
    r = subprocess.run([sys.executable, str(REPO / "apply_embedding_init.py"), "--help"], capture_output=True, text=True, cwd=str(REPO))
    m = re.search(r"No module named '([\w\.]+)'", r.stderr)
    if not m: break
    pkg = m.group(1).split(".")[0]; print("installing", pkg)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", PIPNAME.get(pkg, pkg)], check=True)
print("official script imports OK" if not m else "STILL MISSING: " + r.stderr[-500:])

# ---- compatibility patch (all files of the official repo): recent transformers versions removed AdamW ----
_old = "from transformers import AdamW, get_scheduler"
_new = ("import torch as _torch\nfrom transformers import get_scheduler\n"
        "class AdamW(_torch.optim.AdamW):   # same defaults as the removed transformers.AdamW (weight_decay 0.0, eps 1e-6)\n"
        "    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-6, weight_decay=0.0, **kw): super().__init__(params, lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)")
for _p in REPO.rglob("*.py"):
    _t = _p.read_text()
    if _old in _t: _p.write_text(_t.replace(_old, _new)); print("patched", _p.name)
print([(p.name, l) for p in REPO.rglob("*.py") for l in p.read_text().splitlines() if "import" in l and "AdamW" in l])
# check: the script must now import cleanly (this catches the next missing piece in seconds instead of after 7 minutes of TokDist)
r = subprocess.run([sys.executable, str(REPO / "apply_embedding_init.py"), "--help"], capture_output=True, text=True, cwd=str(REPO))
print("import check:", "OK" if "Traceback" not in r.stderr else r.stderr[-1500:])

# ---- install anything the official script still lacks (pip uses hyphens for some package names) ----
PIPNAME.update({"ahocorasick_rs": "ahocorasick-rs"})
for _ in range(12):
    r = subprocess.run([sys.executable, str(REPO / "apply_embedding_init.py"), "--help"], capture_output=True, text=True, cwd=str(REPO))
    m = re.search(r"No module named '([\w\.]+)'", r.stderr)
    if not m: break
    pkg = m.group(1).split(".")[0]; print("installing", pkg, "->", PIPNAME.get(pkg, pkg.replace("_", "-")))
    q = subprocess.run([sys.executable, "-m", "pip", "install", "-q", PIPNAME.get(pkg, pkg.replace("_", "-"))], capture_output=True, text=True)
    if q.returncode: print(q.stderr[-600:]); break
print("import check:", "OK" if "Traceback" not in r.stderr else r.stderr[-1500:])


In [ ]:
# Environment recorder (called at the end of the notebook)
# ---- Environment record: versions, hardware, checkpoints, data revisions. Writes environment.json and environment_pip_freeze.txt ----
import sys, os, platform, json, subprocess, hashlib, time
from pathlib import Path

def capture_environment(out_dir, model_ids, fasttext_search_roots=()):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    env = dict(captured_utc=time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), python=sys.version.replace("\n", " "),
               platform=platform.platform(), cpu_count=os.cpu_count())
    def pipv(name):
        try:
            from importlib.metadata import version
            return version(name)
        except Exception:
            return None
    names = ["torch", "transformers", "tokenizers", "safetensors", "huggingface_hub", "datasets", "numpy", "scipy", "scikit-learn", "matplotlib",
             "mergekit", "deepfocus", "wechsel", "fasttext", "fasttext-numpy2-wheel", "gensim", "nltk", "accelerate"]
    env["packages"] = {n: pipv(n) for n in names}
    try:
        import torch
        env["torch"] = dict(version=torch.__version__, cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
                            cuda_available=torch.cuda.is_available(), tf32_matmul=torch.backends.cuda.matmul.allow_tf32,
                            tf32_cudnn=torch.backends.cudnn.allow_tf32, cudnn_deterministic=torch.backends.cudnn.deterministic,
                            deterministic_algorithms=torch.are_deterministic_algorithms_enabled(), num_threads=torch.get_num_threads(),
                            gpus=[dict(name=torch.cuda.get_device_name(i), memory_gb=round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1),
                                       capability=list(torch.cuda.get_device_capability(i))) for i in range(torch.cuda.device_count())])
    except Exception as e:
        env["torch"] = f"unavailable: {type(e).__name__}"
    try:
        env["nvidia_smi"] = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                                           capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as e:
        env["nvidia_smi"] = f"unavailable: {type(e).__name__}"
    try:
        fr = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, timeout=120).stdout
        (out_dir / "environment_pip_freeze.txt").write_text(fr); env["pip_freeze_lines"] = len(fr.splitlines())
    except Exception as e:
        env["pip_freeze_lines"] = f"unavailable: {type(e).__name__}"
    # checkpoints: the revision cached on this machine (the one the run loads) and the current hub head
    env["models"] = {}
    try:
        from huggingface_hub import scan_cache_dir, HfApi
        cached = {}
        try:
            for repo in scan_cache_dir().repos:
                if repo.repo_type == "model": cached[repo.repo_id] = sorted(r.commit_hash for r in repo.revisions)
        except Exception:
            pass
        api = HfApi()
        for m in model_ids:
            d = dict(cached_revisions=cached.get(m))
            try: d["hub_head_now"] = api.model_info(m).sha
            except Exception as e: d["hub_head_now"] = f"unavailable: {type(e).__name__}"
            env["models"][m] = d
        env["datasets"] = {}
        for ds in ("qiaojin/PubMedQA", "Salesforce/wikitext"):
            try: env["datasets"][ds] = api.dataset_info(ds).sha
            except Exception as e: env["datasets"][ds] = f"unavailable: {type(e).__name__}"
    except Exception as e:
        env["models"] = f"unavailable: {type(e).__name__}"
    try:
        import urllib.request
        req = urllib.request.Request("https://api.github.com/repos/Franck-Dernoncourt/pubmed-rct/commits/master", headers={"User-Agent": "env-capture"})
        env["datasets"]["Franck-Dernoncourt/pubmed-rct (master head now)"] = json.load(urllib.request.urlopen(req, timeout=30))["sha"]
    except Exception as e:
        env.setdefault("datasets", {})["Franck-Dernoncourt/pubmed-rct (master head now)"] = f"unavailable: {type(e).__name__}"
    # fastText model used by WECHSEL (cc.en.300.bin): size and SHA-256 if present
    env["fasttext_cc_en_300"] = None
    for root in fasttext_search_roots:
        for p in Path(root).rglob("cc.en.300.bin"):
            h = hashlib.sha256()
            with open(p, "rb") as fh:
                for blk in iter(lambda: fh.read(1 << 24), b""): h.update(blk)
            env["fasttext_cc_en_300"] = dict(path=str(p), bytes=p.stat().st_size, sha256=h.hexdigest()); break
        if env["fasttext_cc_en_300"]: break
    (out_dir / "environment.json").write_text(json.dumps(env, indent=2))
    return env



## Text, tokens, terms

In [ ]:
def split_sents(text):
    text = " ".join(text.split())
    return [s for s in re.split(r"(?<=[.!?])\s+", text) if 8 <= len(s.split()) <= 60]

def load_domain():
    if DOMAIN_TXT:
        return [l.strip() for l in open(DOMAIN_TXT, encoding="utf-8") if l.strip()][:N_ABSTRACTS]
    from datasets import load_dataset
    for cfg, stream in (("pqa_artificial", True), ("pqa_labeled", False)):
        try:
            ds, out = load_dataset("qiaojin/PubMedQA", cfg, split="train", streaming=stream), []
            for ex in ds:
                out.append(" ".join(ex["context"]["contexts"]))
                if len(out) >= N_ABSTRACTS: break
            if out: print(f"domain text: PubMedQA/{cfg}, {len(out)} abstracts"); return out
        except Exception as e:
            print(f"PubMedQA/{cfg} failed: {type(e).__name__}: {str(e)[:150]}")
    raise RuntimeError("Could not load PubMedQA. Turn Internet on, or set DOMAIN_TXT to a text file with one abstract per line.")

def load_wiki():
    if not USE_WIKI: return []
    try:
        from datasets import load_dataset
        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
        text = " ".join(x for x in ds["text"][:30000] if x.strip() and not x.startswith(" ="))
        s = split_sents(text); print(f"general control: WikiText-2, {len(s)} sentences"); return s
    except Exception as e:
        print(f"WikiText-2 not available ({type(e).__name__}); only the domain control is used"); return []

SEG = re.compile(r" ?[^\W\d_]+| ?\d+| ?_+| ?[^\s\w]+|\s+")
def segments(s): return SEG.findall(s)

_ENC = {}
def enc(tok, seg):
    k = (getattr(tok, "name_or_path", ""), type(tok).__name__, seg)
    if k not in _ENC: _ENC[k] = tok.encode(seg, add_special_tokens=False)
    return _ENC[k]

def build_seq(segs, tok, term_id, mode):
    ids, fl = [], []
    for g in segs:
        if mode == "new" and g in term_id: ids.append(term_id[g]); fl.append(1)
        else:
            p = enc(tok, g); ids += p; fl += [1 if g in term_id else 0] * len(p)
    return ids, fl

def mk(sents, tok, term_id, need_new):
    segs = [segments(s) for s in sents]
    o = [build_seq(g, tok, term_id, "orig") for g in segs]
    d = dict(sents=sents, o_ids=[x[0] for x in o], o_fl=[x[1] for x in o], n_ids=None, n_fl=None)
    if need_new:
        n = [build_seq(g, tok, term_id, "new") for g in segs]
        d["n_ids"], d["n_fl"] = [x[0] for x in n], [x[1] for x in n]
    return d

def pick_terms(rec_tok, don_tok, train_sents, eval_sents, V0r, V0d):
    """v6: words are chosen from the TRAINING sentences only (eval_sents is accepted but never read).
    Eligible: alphabetic, >= 4 letters, >= MIN_TRAIN_COUNT occurrences in training text, one donor token, >= MIN_PIECES receiver pieces.
    The K_TERMS most frequent in the training text are kept (ties broken alphabetically)."""
    ctr = Counter()
    for s in train_sents: ctr.update(g for g in segments(s)[1:] if re.fullmatch(r" [^\W\d_]{4,}", g))
    out = []
    for g, c in ctr.items():
        if c < MIN_TRAIN_COUNT: continue
        r, d = enc(rec_tok, g), enc(don_tok, g)
        if len(r) >= MIN_PIECES and len(d) == 1 and d[0] < V0d: out.append((c, g, d[0]))
    out.sort(key=lambda x: (-x[0], x[1]))
    return out[:K_TERMS]


## Models, scoring, maps

In [ ]:
def load_receiver(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name).float().to(DEVICE).eval()      # always float32: newer versions load the checkpoint's own precision (OPT is fp16)
    tied = model.get_output_embeddings().weight.data_ptr() == model.get_input_embeddings().weight.data_ptr()
    print(f"  receiver {name}: dtype {next(model.parameters()).dtype}, embeddings {'tied' if tied else 'UNTIED (separate output matrix)'}")
    bos = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id
    return tok, model, bos, tied

def load_donor(name):
    tok = AutoTokenizer.from_pretrained(name)
    m = AutoModelForCausalLM.from_pretrained(name)
    E_in = m.get_input_embeddings().weight.detach().float().cpu().numpy().copy()
    oe = m.get_output_embeddings()
    E_out = None if (oe is None or oe.weight.data_ptr() == m.get_input_embeddings().weight.data_ptr()) else oe.weight.detach().float().cpu().numpy().copy()
    del m; gc.collect(); return tok, E_in, E_out        # E_out is None when the donor ties its embeddings

def make_batch(ids_list, fl_list, idx, bos):
    L = max(len(ids_list[i]) for i in idx) + 1
    x = torch.full((len(idx), L), bos, dtype=torch.long); m = torch.zeros((len(idx), L), dtype=torch.long); f = torch.zeros((len(idx), L - 1))
    for r, i in enumerate(idx):
        s = ids_list[i]; x[r, 1:1 + len(s)] = torch.tensor(s); m[r, :1 + len(s)] = 1; f[r, :len(s)] = torch.tensor(fl_list[i], dtype=torch.float)
    return x.to(DEVICE), m.to(DEVICE), f.to(DEVICE)

def batch_nll(model, x, m, limit=None):
    logits = model(input_ids=x, attention_mask=m).logits[:, :-1].float()
    if limit is not None: logits = logits[:, :, :limit]      # the unchanged receiver: ignore the added tokens completely
    nll = F.cross_entropy(logits.reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1), reduction="none").view(x.size(0), -1)
    return nll, m[:, 1:].float()

@torch.no_grad()
def score(model, bos, ids_list, fl_list, limit=None):
    n = len(ids_list); tot = np.zeros(n); term = np.zeros(n)
    order = sorted(range(n), key=lambda i: len(ids_list[i]))
    for b in range(0, n, EVAL_BATCH):
        idx = order[b:b + EVAL_BATCH]
        x, m, f = make_batch(ids_list, fl_list, idx, bos)
        nll, mm = batch_nll(model, x, m, limit)
        tot[idx] = (nll * mm).sum(1).cpu().numpy(); term[idx] = (nll * mm * f).sum(1).cpu().numpy()
    return tot, term

def set_rows(ctx, rows):
    """rows: array (K, d) for a tied receiver, or a tuple (input_rows, output_rows) for an untied one."""
    items = [(ctx.model.get_input_embeddings().weight, rows[0] if isinstance(rows, tuple) else rows)]
    if isinstance(rows, tuple): items.append((ctx.model.get_output_embeddings().weight, rows[1]))
    with torch.no_grad():
        for w, r in items: w[ctx.V0:ctx.V0 + ctx.K] = torch.as_tensor(np.asarray(r, np.float32), dtype=w.dtype, device=w.device)

def make_rows(ctx, c, d):
    if isinstance(d, tuple): return (ctx.mu_r + c * ctx.rbar * d[0], ctx.mu_ro + c * ctx.rbar_o * d[1])
    return ctx.mu_r + c * ctx.rbar * d

def unit(x, eps=1e-12): return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), eps)

def fit_ridge(Dc, Rc, seed=0):
    s = float(np.linalg.norm(Dc, axis=1).mean()); X = (Dc / s).astype(np.float64); Y = Rc.astype(np.float64)
    perm = np.random.default_rng(seed).permutation(len(X)); nv = min(max(50, len(X) // 10), len(X) // 5); va, tr = perm[:nv], perm[nv:]
    def solve(Xt, Yt, lam):
        A = Xt.T @ Xt; A[np.diag_indices_from(A)] += lam * np.trace(A) / A.shape[0]
        return np.linalg.solve(A, Xt.T @ Yt)
    best = None
    for lam in (1e-4, 1e-3, 1e-2, 1e-1, 1.0):
        cos = (unit(X[va] @ solve(X[tr], Y[tr], lam)) * unit(Y[va])).sum(1).mean()
        if best is None or cos > best[0]: best = (cos, lam)
    return solve(X, Y, best[1]), s, best[1]

def ridge_dirs(Dc, Rc, Dterm_c):
    W, s, lam = fit_ridge(Dc, Rc); return unit((Dterm_c / s) @ W), lam

def omp_dirs(Dc, Rc, Dterm_c, k):
    coef = orthogonal_mp(unit(Dc).astype(np.float64).T, unit(Dterm_c).astype(np.float64).T, n_nonzero_coefs=k)
    return unit(coef.reshape(len(Dc), -1).T @ unit(Rc))

def objective(ctx):
    t, c = ctx.D["t_train"], ctx.D["c_train"]
    tt, _ = score(ctx.model, ctx.bos, t["n_ids"], t["n_fl"]); cc, _ = score(ctx.model, ctx.bos, c["o_ids"], c["o_fl"])
    return 0.5 * tt.sum() / sum(map(len, t["o_ids"])) + 0.5 * cc.sum() / sum(map(len, c["o_ids"]))

def calibrate(ctx, direction):
    best = None
    for c in GRID:
        set_rows(ctx, make_rows(ctx, c, direction)); o = objective(ctx)
        if best is None or o < best[1]: best = (c, o)
    return best

def evaluate(ctx, mode):
    r = {}
    for name in ("t_eval", "c_dom", "c_wiki"):
        S = ctx.D[name]
        if S is None: continue
        use_new = mode == "new" and S["n_ids"] is not None
        ids, fl = (S["n_ids"], S["n_fl"]) if use_new else (S["o_ids"], S["o_fl"])
        tot, term = score(ctx.model, ctx.bos, ids, fl, ctx.V0 if mode == "orig" else None); r[name] = dict(tot=tot, term=term, ntok=np.array(list(map(len, ids))), nbytes=np.array([len(s.encode("utf-8")) for s in S["sents"]]))
    return r

def train_new_rows(ctx, rows, steps, seed):
    set_rows(ctx, rows)
    embs = [ctx.model.get_input_embeddings().weight]
    if not ctx.tied: embs.append(ctx.model.get_output_embeddings().weight)
    backups = [e.detach()[:ctx.V0].clone() for e in embs]
    for p in ctx.model.parameters(): p.requires_grad_(False)
    for e in embs: e.requires_grad_(True)
    opt = torch.optim.AdamW(embs, lr=TRAIN_LR, weight_decay=0.0); ctx.model.eval()
    P = ctx.D["t_pool"]; rng = random.Random(seed); skipped = 0
    for _ in range(steps):
        idx = [rng.randrange(len(P["n_ids"])) for _ in range(BATCH if ctx.model.config.vocab_size < 100000 else 4)]
        x, m, f = make_batch(P["n_ids"], P["n_fl"], idx, ctx.bos)
        nll, mm = batch_nll(ctx.model, x, m); loss = (nll * mm).sum() / mm.sum()
        if not torch.isfinite(loss): skipped += 1; opt.zero_grad(set_to_none=True); continue
        loss.backward()
        for e in embs: e.grad[:ctx.V0] = 0
        torch.nn.utils.clip_grad_norm_(embs, 1.0)
        if not all(bool(torch.isfinite(e.grad).all()) for e in embs): skipped += 1; opt.zero_grad(set_to_none=True); continue
        opt.step(); opt.zero_grad(set_to_none=True)
    for e in embs: e.requires_grad_(False)
    with torch.no_grad():
        for e, b in zip(embs, backups): e[:ctx.V0].copy_(b)        # the original rows are restored exactly, whatever happened
    if skipped: print(f"      note: {skipped} of {steps} training steps skipped (non-finite loss or gradient)")

# ---------- extra baselines (our implementations) ----------
def ls_dirs(Dc, Rc, Dterm_c):
    """plain least squares (lambda -> 0): minimum-norm solution of min ||X W - Y||, then direction only"""
    s = float(np.linalg.norm(Dc, axis=1).mean())
    W = np.linalg.lstsq((Dc / s).astype(np.float64), Rc.astype(np.float64), rcond=None)[0]
    return unit((Dterm_c / s) @ W)

def procrustes_dirs(Dc, Rc, Dterm_c):
    """orthogonal Procrustes between centred spaces; for different dimensions W = U V^T from the SVD of X^T Y (a partial isometry)"""
    s = float(np.linalg.norm(Dc, axis=1).mean())
    M = (Dc / s).astype(np.float64).T @ Rc.astype(np.float64)
    U, _, Vt = np.linalg.svd(M, full_matrices=False)
    return unit((Dterm_c / s) @ (U @ Vt))

def salt_pred(Dc, Rc, Dterm_c, k, lam):
    """SALT-style local regression (OUR ADAPTATION; the original code was not available): for each new token, the k nearest anchors
    in the donor space (cosine), a ridge regression WITH intercept fitted on them (dual form, k x k solve), then the prediction.
    Returns the raw prediction in centred receiver coordinates."""
    s = float(np.linalg.norm(Dc, axis=1).mean()); k = min(k, len(Dc))
    f64 = dict(dtype=torch.float64, device=DEVICE)
    Dn = torch.as_tensor(unit(Dc), **f64); Xa = torch.as_tensor(Dc / s, **f64); Ya = torch.as_tensor(Rc, **f64)
    T = torch.as_tensor(Dterm_c / s, **f64); Tn = torch.as_tensor(unit(Dterm_c), **f64); out = []
    eye = torch.eye(k, **f64)
    for t in range(len(T)):
        idx = torch.topk(Dn @ Tn[t], k).indices
        X, Y = Xa[idx], Ya[idx]; mx, my = X.mean(0), Y.mean(0); Xc, Yc = X - mx, Y - my
        G = Xc @ Xc.T; G = G + lam * torch.trace(G) / k * eye
        out.append(my + ((T[t] - mx) @ Xc.T) @ torch.linalg.solve(G, Yc))
    return torch.stack(out).cpu().numpy()

def official_omp_rows(Dtargets, Dcand, Rcand, k):
    tg = torch.as_tensor(Dtargets, dtype=torch.float32, device=DEVICE); cd = torch.as_tensor(Dcand, dtype=torch.float32, device=DEVICE)
    idx, co = batch_omp(tg, cd, k)
    rc = torch.as_tensor(Rcand, dtype=torch.float32, device=DEVICE)
    return torch.bmm(co.unsqueeze(1), rc[idx]).squeeze(1).cpu().numpy()

def build_side(Dm, Rm, r_ids, d_ids, d_term, f_idx, piece_ids, seed, wech_rows):
    """All methods for ONE embedding matrix (input side, or output side of an untied receiver).
    Dm, Rm: donor / receiver matrices of this side. Returns directions, raw rows, timing, centring constants."""
    R_a, D_a = Rm[r_ids], Dm[d_ids]; mu_r, mu_d = R_a.mean(0), D_a.mean(0); Rc, Dc = R_a - mu_r, D_a - mu_d
    rbar = float(np.linalg.norm(Rc, axis=1).mean()); Dterm_c = Dm[d_term] - mu_d; K = len(d_term)
    dirs, raw, times = {}, {}, {}
    t = time.time(); dirs["ridge[all]"], lam_all = ridge_dirs(Dc, Rc, Dterm_c); times["ridge[all]"] = time.time() - t
    t = time.time(); dirs["ridge[freq5000]"], lam_f = ridge_dirs(Dc[f_idx], Rc[f_idx], Dterm_c); times["ridge[freq5000]"] = time.time() - t
    t = time.time(); dirs["OMP-32"] = omp_dirs(Dc, Rc, Dterm_c, OMP_K); times["OMP-32 (ours, sklearn)"] = time.time() - t
    t = time.time(); omp_shipped = official_omp_rows(Dm[d_term], D_a, R_a, OMP64_K); omp_centred = official_omp_rows(Dterm_c, Dc, Rc, OMP64_K) + mu_r
    times["OMP-64 official (both versions)"] = time.time() - t
    raw["OMP-64 as shipped (raw)"] = omp_shipped; raw["OMP-64 centred (raw)"] = omp_centred
    dirs["OMP-64 centred (cal)"] = unit(omp_centred - mu_r)
    raw["mean-subword (raw)"] = np.stack([Rm[p].mean(0) for p in piece_ids]); dirs["mean-subword (cal)"] = unit(raw["mean-subword (raw)"] - mu_r)
    dirs["random (cal)"] = unit(np.random.default_rng(seed).normal(size=(K, Rm.shape[1])))
    raw["WECHSEL (raw)"] = wech_rows.astype(np.float64); dirs["WECHSEL (cal)"] = unit(wech_rows - mu_r)
    t = time.time(); dirs["LS (cal)"] = ls_dirs(Dc, Rc, Dterm_c); times["LS"] = time.time() - t
    t = time.time(); dirs["Procrustes (cal)"] = procrustes_dirs(Dc, Rc, Dterm_c); times["Procrustes"] = time.time() - t
    t = time.time(); P = salt_pred(Dc, Rc, Dterm_c, SALT_K, SALT_LAM); times["SALT-ours"] = time.time() - t
    dirs["SALT-ours (cal)"] = unit(P); raw["SALT-ours (raw)"] = mu_r + P
    return SimpleNamespace(dirs=dirs, raw=raw, times=times, mu_r=mu_r, rbar=rbar, lam=(lam_all, lam_f))


## Official WECHSEL (same-language mode)

In [ ]:
# WECHSEL is not run in this notebook (it is in the v6 notebook); a placeholder keeps build_side's interface.


## One pair

In [ ]:
# ---------- TokDist: our adaptation of Token Distillation (Dobler, Elliott & de Melo, arXiv:2505.20133; later titled AweDist) ----------
# The paper's idea: learn each new INPUT embedding so that the model's last-layer hidden states, with the new token in place of the word's pieces,
# match the hidden states it produces on the original pieces (MSE), using a few short contexts per word. This is OUR re-implementation from the paper's description
# (initialisation = mean of the word's receiver pieces; 25 contexts per word; hold-out learning-rate choice), NOT the authors' code. Tied receivers only (output embeddings cannot be learned this way).
import collections
def tokdist_rows(ctx, rec_tok, tr_s, terms, term_id, piece_ids, E, seed):
    t0 = time.time(); model, bos, V0, K = ctx.model, ctx.bos, ctx.V0, ctx.K
    rng = random.Random(seed); occ = collections.defaultdict(list)
    for s in tr_s:
        segs = segments(s)
        for i, g in enumerate(segs):
            if g in term_id: occ[term_id[g] - V0].append((segs, i))
    items = []     # (word j, student ids, teacher ids, start position in student, teacher offset)
    for j in range(K):
        pool = occ.get(j, [])
        for segs, i in rng.sample(pool, min(TD_PER_WORD, len(pool))):
            lo = max(0, i - TD_WINDOW); pre = [t for g in segs[lo:i] for t in enc(rec_tok, g)]; suf = [t for g in segs[i + 1:i + 1 + TD_WINDOW] for t in enc(rec_tok, g)]
            pc = piece_ids[j]; items.append((j, [bos] + pre + [V0 + j] + suf, [bos] + pre + list(pc) + suf, 1 + len(pre), len(pc), len(suf)))
    perm = list(range(len(items))); rng.shuffle(perm); nv = max(1, len(items) // 10); val, trn = perm[:nv], perm[nv:]
    # teacher hidden states (original pieces), fixed
    def pad(seqs):
        L = max(map(len, seqs)); x = torch.full((len(seqs), L), bos, dtype=torch.long); m = torch.zeros((len(seqs), L), dtype=torch.long)
        for r, s_ in enumerate(seqs): x[r, :len(s_)] = torch.tensor(s_); m[r, :len(s_)] = 1
        return x.to(DEVICE), m.to(DEVICE)
    tgt = [None] * len(items)
    with torch.no_grad():
        for b in range(0, len(items), 64):
            ids = list(range(b, min(b + 64, len(items)))); x, m = pad([items[i][2] for i in ids])
            h = model(input_ids=x, attention_mask=m, output_hidden_states=True).hidden_states[-1]
            for r, i in enumerate(ids):
                _, _, _, p0, mlen, slen = items[i]; tgt[i] = h[r, p0 + mlen - 1:p0 + mlen + slen].float().cpu()
    emb = model.get_input_embeddings()
    init = np.stack([E[list(piece_ids[j])].mean(0) for j in range(K)]).astype(np.float32)
    def student_loss(new, ids):
        seqs = [items[i][1] for i in ids]; x, m = pad(seqs)
        isnew = x >= V0; base = emb(torch.where(isnew, torch.zeros_like(x), x)); sel = (x - V0).clamp(min=0)
        newv = new[sel]; ie = torch.where(isnew.unsqueeze(-1), newv, base)
        h = model(inputs_embeds=ie, attention_mask=m, output_hidden_states=True).hidden_states[-1]
        loss, n = 0.0, 0
        for r, i in enumerate(ids):
            _, _, _, p0, mlen, slen = items[i]; t = tgt[i].to(DEVICE); loss = loss + ((h[r, p0:p0 + 1 + slen] - t) ** 2).mean(1).sum(); n += t.shape[0]
        return loss / n
    BSZ = TD_BATCH if E.shape[1] <= 1280 else 8      # smaller batch for wide receivers (memory)
    for p in model.parameters(): p.requires_grad_(False)
    best = None
    for lr in TD_LRS:
        new = torch.nn.Parameter(torch.tensor(init, device=DEVICE)); opt = torch.optim.AdamW([new], lr=lr, weight_decay=0.0); order = list(trn)
        for ep in range(TD_EPOCHS):
            rng.shuffle(order)
            for b in range(0, len(order), BSZ):
                loss = student_loss(new, order[b:b + BSZ]); opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad(): v = float(sum(student_loss(new, val[b:b + 64]).item() * len(val[b:b + 64]) for b in range(0, len(val), 64)) / len(val))
        print(f"    TokDist lr {lr:g}: held-out hidden-state MSE {v:.4f}")
        if best is None or v < best[0]: best = (v, lr, new.detach().cpu().numpy().copy())
    for p in model.parameters(): p.requires_grad_(False)
    print(f"    TokDist: {len(items)} contexts ({len(trn)} train / {len(val)} held out), chose lr {best[1]:g}, {time.time() - t0:.0f}s")
    return best[2], time.time() - t0


In [ ]:
# ---------- Official AweDist: run the authors' script on OUR words and OUR training abstracts, then read the learned rows ----------
def awedist_official_rows(pair, rec_tok, terms, train_abstracts, piece_ids, E):
    """Runs `apply_embedding_init.py --init_method=awedist` (authors' code, wordlist mode) with our word list and our training abstracts as the corpus.
    Words for which the corpus has too few snippets are dropped by the script; they keep the script's own start point (mean of their pieces) here and are counted.
    The receiver is first saved as a float32 copy so that the script trains in float32 (OPT's checkpoint is float16)."""
    t0 = time.time(); K = len(terms); name = pair["receiver"]; tag = name.replace("/", "_")
    rec_dir = WORK / f"{tag}_fp32"
    if not rec_dir.exists():
        m = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float32); m.save_pretrained(rec_dir); AutoTokenizer.from_pretrained(name).save_pretrained(rec_dir); del m
    corpus = REPO / f"train_{DOMAIN}"; corpus.mkdir(exist_ok=True)
    with open(corpus / "train.jsonl", "w", encoding="utf-8") as f:
        for a in train_abstracts: f.write(json.dumps({"text": a}) + "\n")
    import glob as _g                                      # the script caches the tokenised corpus by name; drop old caches so that it sees THIS corpus (7500 documents)
    for _p in _g.glob(os.path.expanduser(os.getenv("XDG_CACHE_HOME", "~/.cache")) + "/tokenized_datasets/*"): shutil.rmtree(_p, ignore_errors=True)
    words_json = WORK / f"words_{tag}_{pair['donor'].replace('/', '_')}.json"; words_json.write_text(json.dumps({"words": [g for _, g, _d in terms]}))
    out = WORK / f"out_{tag}_{pair['donor'].replace('/', '_')}"
    if out.exists(): shutil.rmtree(out)
    cmd = [sys.executable, "apply_embedding_init.py", f"--model_path={rec_dir}", f"--out_path={out}", "--build_target_method=wordlist", f"--new_tokens_source={words_json}",
           f"--dataset_path=train_{DOMAIN}", "--init_method=awedist", f"--awedist_lr={AW_LR}", f"--awedist_epochs={AW_EPOCHS}", f"--snippet_num={AW_SNIPPETS}", "--mixed_precision=False",
           "--stopping_condition=num_docs:3000"]    # the script reads documents in growing batches (500, 1000, 2000, ...) and fails when the corpus runs out; this stops it after 3500 documents
    print("    official AweDist:", " ".join(cmd[1:])); r = subprocess.run(cmd, cwd=str(REPO), capture_output=True, text=True)
    (WORK / f"log_{tag}_{pair['donor'].replace('/', '_')}.txt").write_text(r.stdout + "\n" + r.stderr)
    if r.returncode != 0 or not out.exists():
        print(r.stdout[-2500:]); print(r.stderr[-3500:]); raise RuntimeError("official AweDist script failed (output above)")
    for l in r.stdout.splitlines():
        if any(k in l for k in ("Enough snippets", "Not enough snippets", "potential tokens", "filtered tokens", "Found ")): print("    " + l)
    tk = AutoTokenizer.from_pretrained(out); added = tk.get_added_vocab()
    mdl = AutoModelForCausalLM.from_pretrained(out, torch_dtype=torch.float32); W = mdl.get_input_embeddings().weight.detach().cpu().numpy(); del mdl
    rows = np.stack([E[list(piece_ids[j])].mean(0) for j in range(K)]).astype(np.float32); got = 0
    for j, (_, g, _d) in enumerate(terms):
        tid = added.get(g)
        if tid is not None: rows[j] = W[tid]; got += 1
    print(f"    official AweDist: learned rows for {got} of {K} words (the rest keep the mean-of-pieces start); {time.time() - t0:.0f}s")
    return rows, time.time() - t0, got


_EXTRA = []
def aw_corpus(train_abstracts):
    """Our training abstracts + extra PubMedQA abstracts (positions after the first N_ABSTRACTS, never used for evaluation) up to N_AW_DOCS documents."""
    need = N_AW_DOCS - len(train_abstracts)
    if need > 0 and len(_EXTRA) < need and not DOMAIN_TXT:
        from datasets import load_dataset
        ds = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train", streaming=True)
        for i, ex in enumerate(ds):
            if i < N_ABSTRACTS: continue
            _EXTRA.append(" ".join(ex["context"]["contexts"]))
            if len(_EXTRA) >= need: break
    return (list(train_abstracts) + _EXTRA[:max(need, 0)])[:N_AW_DOCS]


In [ ]:
# ---- AweDist hooks for the official script: (1) stop after 3500 documents; (2) give the tokenizer a pad token that is NOT the end-of-text token ----
# (the script drops every token equal to the pad id; GPT-2 has no pad, so it uses end-of-text, which is also the start token of every snippet -> tensors of different length)
import importlib, subprocess, json, os
importlib.reload(subprocess)          # undo any earlier wrapper, so this cell can be run again
def _fix_pad(model_dir):
    tk = AutoTokenizer.from_pretrained(model_dir); special = {tk.eos_token_id, tk.bos_token_id}
    if tk.pad_token_id is not None and tk.pad_token_id not in special: return
    used = set()
    for l in open(REPO / f"train_{DOMAIN}" / "train.jsonl", encoding="utf-8"): used.update(tk(json.loads(l)["text"], add_special_tokens=False)["input_ids"])
    pid = next(i for i in range(len(tk) - 1, -1, -1) if i not in used and i not in special)
    tk.pad_token = tk.convert_ids_to_tokens(pid); tk.save_pretrained(model_dir); print(f"    pad token set to id {pid} ({tk.pad_token!r}), not used in our text")
def _make(orig):
    def _run(cmd, *a, **k):
        if isinstance(cmd, list) and any("apply_embedding_init.py" in str(x) for x in cmd) and "--help" not in cmd:
            cmd = [x for x in cmd if not str(x).startswith("--stopping_condition")] + ["--stopping_condition=num_docs:3000"]
            _fix_pad(next(str(x).split("=", 1)[1] for x in cmd if str(x).startswith("--model_path=")))
        if isinstance(cmd, list) and any("apply_embedding_init.py" in str(x) for x in cmd) and "--help" not in cmd and torch.cuda.device_count() > 1:
            k["env"] = {**os.environ, "CUDA_VISIBLE_DEVICES": "1"}      # the notebook holds GPU 0; run the official script on the second GPU
        return orig(cmd, *a, **k)
    return _run
subprocess.run = _make(subprocess.run)


In [ ]:
def run_pair(pair, seed=0):
    t_all = time.time(); name = pair["name"]
    rec_tok, model, bos, tied = load_receiver(pair["receiver"]); don_tok, D_in, D_out = load_donor(pair["donor"])
    V0 = model.get_input_embeddings().weight.shape[0]; V0d = D_in.shape[0]
    E = model.get_input_embeddings().weight.detach().float().cpu().numpy().copy()
    E_out = None if tied else model.get_output_embeddings().weight.detach().float().cpu().numpy().copy()
    abstracts = load_domain(); n_tr = int(len(abstracts) * TRAIN_FRAC)
    tr_s = [s for a in abstracts[:n_tr] for s in split_sents(a)]; ev_s = [s for a in abstracts[n_tr:] for s in split_sents(a)]
    terms = pick_terms(rec_tok, don_tok, tr_s, ev_s, V0, V0d)
    (OUT / f"words_{DOMAIN}__{pair['name'].replace(' ', '_').replace('<-', 'from')}.json").write_text(json.dumps([g for _, g, _d in terms]))   # word list frozen to disk before any sentence is scored
    print(f"{name}: {'tied' if tied else 'UNTIED'} receiver dim {E.shape[1]}, donor dim {D_in.shape[1]} | eligible new words (donor 1 token, receiver >= {MIN_PIECES} pieces): {len(terms)}")
    if len(terms) < MIN_TERMS: print(f"  fewer than {MIN_TERMS} eligible words; skipped"); return None
    for _, g, _d in terms[:12]: print(f"    {g!r:<18} receiver pieces: {[rec_tok.convert_ids_to_tokens(i) for i in enc(rec_tok, g)]}")
    K = len(terms); term_id = {g: V0 + j for j, (_, g, _d) in enumerate(terms)}
    d_term = np.array([d for _, _g, d in terms]); piece_ids = [enc(rec_tok, g) for _, g, _d in terms]
    # anchors: whole words that are single tokens in both models (matched by token string)
    rv, dv = rec_tok.get_vocab(), don_tok.get_vocab()
    keys = sorted((k for k in rv if k in dv and k.startswith("Ġ") and k[1:].isalpha() and len(k) >= 3 and rv[k] < V0 and dv[k] < V0d), key=lambda k: rv[k])
    r_ids = np.array([rv[k] for k in keys]); d_ids = np.array([dv[k] for k in keys]); print(f"  shared anchor words: {len(keys)}")
    cnt = Counter(i for s in tr_s for g in segments(s) if g not in term_id for i in enc(rec_tok, g))
    f_idx = np.argsort([-cnt[i] for i in r_ids], kind="stable")[:min(5000, len(keys))]
    # sentence sets
    rng = random.Random(seed); has = lambda s: any(g in term_id for g in segments(s))
    tr_t = [s for s in tr_s if has(s)]; tr_c = [s for s in tr_s if not has(s)]
    ev_t = [s for s in ev_s if has(s)]; ev_c = [s for s in ev_s if not has(s)]
    wiki = [s for s in load_wiki() if not has(s)]
    smp = lambda L, n: rng.sample(L, min(n, len(L)))
    c_train_s = smp(tr_c, N_TRAIN_C // 2 if wiki else N_TRAIN_C) + (smp(wiki[:len(wiki) // 2], N_TRAIN_C // 2) if wiki else [])
    D = dict(t_train=mk(smp(tr_t, N_TRAIN_T), rec_tok, term_id, True), c_train=mk(c_train_s, rec_tok, term_id, False),
             t_pool=mk(smp(tr_t, POOL_MAX), rec_tok, term_id, True), t_eval=mk(smp(ev_t, N_EVAL_T), rec_tok, term_id, True),
             c_dom=mk(smp(ev_c, N_EVAL_C), rec_tok, term_id, False),
             c_wiki=mk(smp(wiki[len(wiki) // 2:], N_EVAL_C), rec_tok, term_id, False) if wiki else None)
    print(f"  sentences: calibration {len(D['t_train']['sents'])}+{len(D['c_train']['sents'])} | pool {len(D['t_pool']['sents'])} | eval with new words {len(D['t_eval']['sents'])}, domain control {len(D['c_dom']['sents'])}, wiki control {len(D['c_wiki']['sents']) if D['c_wiki'] else 0}")
    model.resize_token_embeddings(V0 + K)
    same = model.get_output_embeddings().weight.data_ptr() == model.get_input_embeddings().weight.data_ptr()
    if same != tied: raise RuntimeError("tie status changed after resize")
    # ---- all methods, input side (and output side for an untied receiver) ----
    W_in, w_fb, wt = np.zeros((V0d, E.shape[1]), np.float32), set(), 0.0      # WECHSEL not run here
    S_in = build_side(D_in, E, r_ids, d_ids, d_term, f_idx, piece_ids, seed, W_in[d_term]); S_in.times["WECHSEL official (first call per pair builds the whole donor vocabulary)"] = wt
    n_fb = sum(don_tok.convert_ids_to_tokens(int(i)) in w_fb for i in d_term); print(f"  WECHSEL: {n_fb} of {K} new words had no fastText vector (random fallback)")
    S_out = None
    if not tied: raise RuntimeError('learned baselines are defined for tied receivers only')
    if not tied:
        Dm_out = D_out if D_out is not None else D_in         # a tied donor has one matrix for both roles
        W_out, _ = wechsel_matrix(pair, rec_tok, don_tok, E_out, "out")
        S_out = build_side(Dm_out, E_out, r_ids, d_ids, d_term, f_idx, piece_ids, seed + 1, W_out[d_term])
    comb = (lambda a, b: a) if tied else (lambda a, b: (a, b))
    ctx = SimpleNamespace(model=model, bos=bos, V0=V0, K=K, D=D, tied=tied, mu_r=S_in.mu_r, rbar=S_in.rbar,
                          mu_ro=None if tied else S_out.mu_r, rbar_o=None if tied else S_out.rbar)
    dirs = {k: comb(v, None if tied else S_out.dirs[k]) for k, v in S_in.dirs.items()}
    raws = {k: comb(v, None if tied else S_out.raw[k]) for k, v in S_in.raw.items()}
    KEEP = ("OMP-64 centred (cal)", "ridge[all]", "ridge[freq5000]"); dirs = {k: v for k, v in dirs.items() if k in KEEP}
    times = dict(S_in.times); lam_all, lam_f = S_in.lam
    print(f"  map times (s) for {K} words, input side: " + ", ".join(f"{k} {v:.2f}" for k, v in times.items()) + f" | ridge lambda all {lam_all}, freq {lam_f}")
    rows, scales, objs, calib_s = dict(raws), {}, {}, {}
    rows["TokDist (ours)"], td_s = tokdist_rows(ctx, rec_tok, tr_s, terms, term_id, piece_ids, E, seed); calib_s["TokDist (ours)"] = td_s
    rows["AweDist (official)"], aw_s, aw_got = awedist_official_rows(pair, rec_tok, terms, aw_corpus(abstracts[:n_tr]), piece_ids, E); calib_s["AweDist (official)"] = aw_s; aw_info = dict(covered=aw_got, K=K, commit=AW_COMMIT, lr=AW_LR, snippet_num=AW_SNIPPETS, epochs=AW_EPOCHS, secs=aw_s, n_docs=N_AW_DOCS)
    dirs["TokDist (ours, cal)"] = unit(rows["TokDist (ours)"] - ctx.mu_r); dirs["AweDist (official, cal)"] = unit(rows["AweDist (official)"] - ctx.mu_r)   # same direction, length chosen like for ridge/OMP
    for cond, d in dirs.items():
        _t = time.time(); c, o = calibrate(ctx, d); calib_s[cond] = time.time() - _t; rows[cond] = make_rows(ctx, c, d); scales[cond] = c; objs[cond] = o
        print(f"  {cond}: scale {c}{'  (EDGE)' if c in (GRID[0], GRID[-1]) else ''} | train objective {o:.4f}")
    sel = "ridge[freq5000]" if objs["ridge[freq5000]"] < objs["ridge[all]"] else "ridge[all]"
    print(f"  ridge variant chosen by the training objective: {sel}")
    CONDS = ["original", "AweDist (official)", "AweDist (official, cal)", "TokDist (ours)", "TokDist (ours, cal)", "OMP-64 centred (cal)", "ridge[all]", "ridge[freq5000]"]
    curves = {}
    for cond in CONDS:
        t1 = time.time(); curves[cond] = {}
        if cond != "original": set_rows(ctx, rows[cond])
        curves[cond][0] = evaluate(ctx, "orig" if cond == "original" else "new")
        if TRAIN_STEPS > 0 and cond != "original":
            train_new_rows(ctx, rows[cond], TRAIN_STEPS, seed); curves[cond][TRAIN_STEPS] = evaluate(ctx, "new")
        a = curves[cond][0]["t_eval"]["tot"].mean()
        print(f"    {cond:<20} term-sentence NLL {a:.2f}" + (f" -> {curves[cond][TRAIN_STEPS]['t_eval']['tot'].mean():.2f} after {TRAIN_STEPS} steps" if TRAIN_STEPS > 0 and cond != "original" else "") + f"   [{time.time() - t1:.0f}s]")
    dmax = float((model.get_input_embeddings().weight[:V0].detach().float().cpu() - torch.as_tensor(E)).abs().max())
    if not tied: dmax = max(dmax, float((model.get_output_embeddings().weight[:V0].detach().float().cpu() - torch.as_tensor(E_out)).abs().max()))
    print(f"  max change in the original embedding rows: {dmax:.3g}")
    assert dmax == 0.0, f"old embedding rows changed by {dmax:.3g}"
    res = dict(pair=name, domain=DOMAIN, seed=seed, K=K, terms=[g for _, g, _d in terms], dims=(E.shape[1], D_in.shape[1]), n_anchors=len(keys), curves=curves, scales=scales, objs=objs,
               sel=sel, tied=tied, wechsel_fallback=n_fb, map_s=times, calib_s=calib_s, minutes=(time.time() - t_all) / 60, train_steps=TRAIN_STEPS, awedist=aw_info)
    print(f"  done in {res['minutes']:.1f} min")
    del ctx, model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return res

## Run (resumable). Pairs that fail are reported and skipped.

In [ ]:
for SEED in SEEDS:
  for pair in PAIRS:
      f = OUT / (f"{DOMAIN}__" + pair["name"].replace(" ", "_").replace("<-", "from").replace(" ", "") + f"__seed{SEED}.pkl")
      if f.exists(): print("already done:", pair["name"]); continue
      print(f"\n########## {pair['name']} | seed {SEED} ##########")
      try:
          res = run_pair(pair, SEED)
          if res is not None:
              with open(f, "wb") as fh: pickle.dump(res, fh)
      except Exception as e:
          print(f"!!! failed: {type(e).__name__}: {e}"); traceback.print_exc()
      gc.collect()
      if DEVICE == "cuda": torch.cuda.empty_cache()

## Report

In [ ]:
def boot(d, n_boot=2000):
    rng = np.random.default_rng(0); n = len(d); m = d[rng.integers(0, n, (n_boot, n))].mean(1)
    return float(d.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

LN2 = float(np.log(2))
def bpb(a, b):
    """difference in bits per byte, a minus b: mean per-sentence NLL difference (nats) / ln 2 / mean sentence bytes, with a paired bootstrap interval"""
    d = (a["tot"] - b["tot"]) / LN2 / a["nbytes"].mean(); m, lo, hi = boot(d)
    return f"{m:+.4f} [{lo:+.4f}, {hi:+.4f}]{'*' if lo > 0 or hi < 0 else ' '}"

def fmt(d):
    m, lo, hi = boot(d); return f"{m:+.2f} [{lo:+.2f}, {hi:+.2f}]{'*' if lo > 0 or hi < 0 else ' '}"

for f in sorted(OUT.glob("*.pkl")):
    with open(f, "rb") as fh: r = pickle.load(fh)
    cv, sel = r["curves"], r["sel"]; steps = [0] + ([r["train_steps"]] if r["train_steps"] > 0 else [])
    print(f"\n{'=' * 100}\n{r['pair']} | seed {r['seed']} | {r['K']} new words | dims receiver {r['dims'][0]} / donor {r['dims'][1]} | {r['n_anchors']} anchor words | chosen ridge: {sel}\n{'=' * 100}")
    o = cv["original"][0]
    print(f"embeddings: {'tied' if r.get('tied', True) else 'UNTIED (input and output rows both initialised)'} | mean bytes per evaluation sentence: {o['t_eval']['nbytes'].mean():.1f}")
    print(f"tokens per sentence containing new words: {o['t_eval']['ntok'].mean():.1f} (words split) -> {cv['ridge[all]'][0]['t_eval']['ntok'].mean():.1f} (new tokens)")
    for st in steps:
        print(f"\n{'zero-shot' if st == 0 else f'after {st} steps training only the new rows'}  (differences vs the unchanged receiver, nats per sentence; negative = better; * = CI excludes 0)")
        print(f"{'method':<22}{'sentence NLL':>13}  {'diff vs original':<26}{'diff, new words only':<26}{'wiki control':>14}{'domain control':>16}")
        for c in [c for c in ["mean-subword (raw)", "mean-subword (cal)", "random (cal)", "OMP-32", "OMP-64 as shipped (raw)", "OMP-64 centred (raw)", "OMP-64 centred (cal)", "WECHSEL (raw)", "WECHSEL (cal)", "LS (cal)", "Procrustes (cal)", "SALT-ours (raw)", "SALT-ours (cal)", "ridge[all]", "ridge[freq5000]"] if c in cv]:
            x = cv[c][st]; line = f"{c + (' *' if c == sel else ''):<22}{x['t_eval']['tot'].mean():>13.2f}  {fmt(x['t_eval']['tot'] - o['t_eval']['tot']):<26}{fmt(x['t_eval']['term'] - o['t_eval']['term']):<26}"
            for cs in ("c_wiki", "c_dom"):
                line += f"{(x[cs]['tot'] - o[cs]['tot']).mean():>+14.2f}" if cs in x else f"{'n/a':>14}"
            print(line)
        print("  direct comparisons (sentence NLL difference):")
        for a, b in [(a, b) for a, b in ((sel, "OMP-32"), (sel, "OMP-64 centred (raw)"), (sel, "OMP-64 centred (cal)"), (sel, "OMP-64 as shipped (raw)"), (sel, "WECHSEL (raw)"), (sel, "WECHSEL (cal)"), (sel, "mean-subword (cal)"), (sel, "mean-subword (raw)"), (sel, "LS (cal)"), (sel, "Procrustes (cal)"), (sel, "SALT-ours (raw)"), (sel, "SALT-ours (cal)"), ("OMP-32", "mean-subword (cal)")) if b in cv]:
            print(f"    {a} - {b}: {fmt(cv[a][st]['t_eval']['tot'] - cv[b][st]['t_eval']['tot'])}   |  bits/byte: {bpb(cv[a][st]['t_eval'], cv[b][st]['t_eval'])}")
    print(f"\nmap time for {r['K']} words (s): " + ", ".join(f"{k} {v:.2f}" for k, v in r["map_s"].items()))

## Record the environment
Writes `environment.json` (versions, GPU, checkpoint revisions, fastText hash) next to the results.

In [ ]:
_ids = sorted({p[k] for p in PAIRS for k in ('receiver', 'donor')})
try:
    capture_environment(OUT, _ids, [CACHE_ROOT])
    print('environment recorded in', OUT / 'environment.json')
except Exception as e:
    print('environment record failed:', type(e).__name__, e)
